In [1]:
import sys

print(sys.executable)

c:\Users\Mirza Shareef Baig\OneDrive\Desktop\Healthrisk-ai\.venv\Scripts\python.exe


In [2]:
import sys

!"{sys.executable}" -m pip install lightgbm shap pytest

In [3]:
from lightgbm import LGBMRegressor

print("✅ LightGBM Installed Successfully")

✅ LightGBM Installed Successfully


In [4]:
import sys

!"{sys.executable}" -m pip install lightgbm shap pytest

In [5]:
from lightgbm import LGBMRegressor

print("✅ LightGBM Installed Successfully")

✅ LightGBM Installed Successfully


In [6]:
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

from utils.feature_engineering import create_features

df = pd.read_csv(
    PROJECT_ROOT / "data" / "raw" / "patient_financial_risk.csv"
)

df = create_features(df)

print(df.shape)
df.head()

(10000, 23)


,patient_id,age,gender,bmi,smoker,diabetes,hypertension,heart_disease,hospital_visits,medication_count,...,hospital_debt_ratio,pharma_investment_score,esg_score,pandemic_risk_index,future_health_cost,risk_level,chronic_condition_count,claims_to_premium_ratio,health_risk_score,high_cost_patient
0,1,69,Female,25.3,0,0,1,1,1,3,...,0.29,19.69,84.55,0.511,27769.38,High,2,2.001611,6.639,1
1,2,32,Male,31.0,0,0,0,0,1,1,...,0.42,2.03,66.94,0.801,13585.13,Low,0,1.987047,2.070,0
2,3,89,Male,39.2,0,0,1,0,1,3,...,0.41,27.45,61.35,0.410,17540.43,Medium,1,1.424980,5.456,1
3,4,78,Female,25.2,0,1,0,1,4,5,...,0.58,95.81,67.46,0.275,40978.02,High,2,2.828050,8.316,1
4,5,38,Female,36.3,0,0,0,0,2,1,...,0.53,57.89,35.44,0.942,14812.69,Medium,0,1.979055,2.849,0


In [7]:
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

from sklearn.ensemble import StackingRegressor
from sklearn.linear_model import Ridge

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

In [8]:
xgb_model = XGBRegressor(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    random_state=42
)

lgbm_model = LGBMRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    random_state=42
)

ensemble_model = StackingRegressor(
    estimators=[
        ("xgb", xgb_model),
        ("lgbm", lgbm_model)
    ],
    final_estimator=Ridge(alpha=1.0)
)

models = {
    "XGBoost": xgb_model,
    "LightGBM": lgbm_model,
    "Stacking Ensemble": ensemble_model
}

In [10]:
results = []

for name, model in models.items():

    print(f"Training {name}...")

    model.fit(X_train, y_train)

    predictions = model.predict(X_test)

    mae = mean_absolute_error(y_test, predictions)
    rmse = mean_squared_error(y_test, predictions) ** 0.5
    r2 = r2_score(y_test, predictions)

    results.append({
        "Model": name,
        "MAE": round(mae, 2),
        "RMSE": round(rmse, 2),
        "R2": round(r2, 4)
    })

results_df = pd.DataFrame(results)

results_df

Training XGBoost...


NameError: name 'X_train' is not defined

In [11]:
from sklearn.model_selection import train_test_split

X = df.drop(
    columns=[
        "patient_id",
        "future_health_cost",
        "risk_level"
    ]
)

X = pd.get_dummies(X, drop_first=True)

y = df["future_health_cost"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("Training shape:", X_train.shape)
print("Testing shape :", X_test.shape)

Training shape: (8000, 20)
Testing shape : (2000, 20)


In [12]:
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from sklearn.ensemble import StackingRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import pandas as pd

xgb_model = XGBRegressor(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    random_state=42
)

lgbm_model = LGBMRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    random_state=42
)

ensemble_model = StackingRegressor(
    estimators=[
        ("xgb", xgb_model),
        ("lgbm", lgbm_model)
    ],
    final_estimator=Ridge(alpha=1.0)
)

models = {
    "XGBoost": xgb_model,
    "LightGBM": lgbm_model,
    "Stacking Ensemble": ensemble_model
}

results = []

for name, model in models.items():
    print(f"Training {name}...")
    model.fit(X_train, y_train)
    preds = model.predict(X_test)

    results.append({
        "Model": name,
        "MAE": round(mean_absolute_error(y_test, preds), 2),
        "RMSE": round(mean_squared_error(y_test, preds) ** 0.5, 2),
        "R2": round(r2_score(y_test, preds), 4)
    })

results_df = pd.DataFrame(results)
results_df

Training XGBoost...
Training LightGBM...
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001243 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2233
[LightGBM] [Info] Number of data points in the train set: 8000, number of used features: 20
[LightGBM] [Info] Start training from score 18679.577224
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No f

,Model,MAE,RMSE,R2
0,XGBoost,1323.86,1809.21,0.9610
1,LightGBM,1316.88,1795.61,0.9616
2,Stacking Ensemble,1316.53,1791.19,0.9617


In [13]:
import joblib

reports_dir = PROJECT_ROOT / "reports"
trained_dir = PROJECT_ROOT / "models" / "trained"

reports_dir.mkdir(exist_ok=True)
trained_dir.mkdir(exist_ok=True)

results_df.to_csv(reports_dir / "model_comparison_results.csv", index=False)

joblib.dump(ensemble_model, trained_dir / "stacking_ensemble_cost_model.pkl")

print("Saved model comparison and ensemble model.")

Saved model comparison and ensemble model.


In [14]:
from explainability.counterfactual import (
    generate_counterfactuals,
    format_counterfactual_summary
)

sample_patient = X_test.iloc[[0]]

result = generate_counterfactuals(
    model=ensemble_model,
    patient_row=sample_patient,
    feature_columns=list(X_test.columns),
    target_reduction=0.10
)

result["recommendations"][:3]

[{'feature_changed': 'claims_to_premium_ratio',
  'change_applied': -0.1,
  'baseline_cost': 44127.12,
  'new_cost': 43429.63,
  'estimated_savings': 697.49,
  'estimated_savings_pct': 1.58,
  'meets_target': False},
 {'feature_changed': 'health_risk_score',
  'change_applied': -0.5,
  'baseline_cost': 44127.12,
  'new_cost': 43965.15,
  'estimated_savings': 161.98,
  'estimated_savings_pct': 0.37,
  'meets_target': False},
 {'feature_changed': 'pandemic_risk_index',
  'change_applied': -0.1,
  'baseline_cost': 44127.12,
  'new_cost': 44266.58,
  'estimated_savings': -139.45,
  'estimated_savings_pct': -0.32,
  'meets_target': False}]

In [15]:
print(format_counterfactual_summary(result))

Baseline predicted healthcare cost is $44,127.12. The strongest actionable scenario is changing claims_to_premium_ratio by -0.1, which may reduce predicted cost to $43,429.63, an estimated savings of $697.49 (1.58%).


In [16]:
from explainability.report_generator import ExplainabilityReport

report = ExplainabilityReport()

report.add_section(
    "Partial Dependence",
    {
        "features": [
            "age",
            "bmi",
            "hospital_visits",
            "annual_claim_amount"
        ],
        "plots_generated": 8
    }
)

report.save_json()

print(report.summary())

{'generated': '2026-06-26T15:41:31.752395', 'modules': ['Partial Dependence']}
